# **Mandatory Assignment 2**
## *Natural Language Processing and Text Analytics (KAN-CDSCO1002U)*
*Group: *

*Student IDs: 185912, 161989, 160714 & 160363*

*Dataset: CyberBullying Comments Dataset*

# Part 1: Extended Abstract - Lexicon-Based Sentiment Analysis

## Lexicon-Based Sentiment Analysis: Foundations, Applications, and Limitations

### Introduction

Sentiment analysis refers to the computational task of identifying and extracting subjective information from text, typically classifying expressed opinions as positive, negative, or neutral (Liu, 2012). Among the established approaches, lexicon-based sentiment analysis stands out for its transparency, interpretability, and independence from labeled training data. Rather than learning sentiment patterns from annotated corpora, lexicon-based methods rely on predefined dictionaries that map words or phrases to sentiment scores. This extended abstract outlines the foundational principles of lexicon-based sentiment analysis, surveys its application across domains, and discusses its strengths and limitations relative to machine learning approaches.

### Foundations and Mechanism

A sentiment lexicon is a structured resource that associates words with polarity values. Well-known examples include SentiWordNet (Baccianella et al., 2010), the AFINN lexicon (Nielsen, 2011), and VADER (Hutto & Gilbert, 2014). The core mechanism is straightforward: given an input text, each token is matched against the lexicon, and the sentiment of the document is computed as an aggregation of individual word scores.

VADER (Valence Aware Dictionary and sEntiment Reasoner) extends this basic approach by incorporating grammatical and syntactical heuristics. It accounts for intensifiers ("very good"), negations ("not bad"), punctuation emphasis, and capitalization, making it particularly suited for informal text such as social media posts (Hutto & Gilbert, 2014). AFINN, by contrast, assigns integer scores ranging from -5 to +5 to approximately 2,477 English words, offering a simpler but effective scoring system (Nielsen, 2011).

The general workflow involves (1) text preprocessing, including tokenization and optional lowercasing, (2) token-level lookup against the lexicon, and (3) score aggregation at the document level, often through summation or averaging.

### Applications Across Domains

Lexicon-based sentiment analysis has been applied extensively across domains where labeled data is scarce or where interpretability is paramount.

In **financial text mining**, Loughran and McDonald (2011) demonstrated that general-purpose sentiment lexicons perform poorly on financial texts because words like "liability" or "risk" carry negative connotations in everyday language but are neutral in financial contexts. Their domain-specific lexicon has since become a standard resource for analyzing earnings calls, 10-K filings, and financial news.

In **social media monitoring**, VADER has been widely adopted for analyzing Twitter data due to its sensitivity to informal language features such as slang, emoticons, and abbreviation patterns (Hutto & Gilbert, 2014). Public health researchers have used lexicon-based tools to track sentiment toward vaccination campaigns and disease outbreaks in real time (Salathé & Khandelwal, 2011).

In **product and service reviews**, lexicon-based approaches serve as baselines for opinion mining systems. Taboada et al. (2011) proposed the Semantic Orientation Calculator (SO-CAL), which applies intensification and negation rules on top of a manually curated lexicon to achieve competitive performance on review datasets.

### Strengths and Limitations

The primary strength of lexicon-based methods is their **transparency**. Each sentiment score can be traced back to specific words and rules, which supports human interpretability and domain validation. Additionally, these methods require **no labeled training data**, making them immediately applicable to new domains or languages provided a suitable lexicon exists.

However, lexicon-based methods face notable limitations. They struggle with **context-dependent meaning**, such as sarcasm and irony, where surface-level word polarity diverges from intended sentiment. They are also sensitive to **domain mismatch**, as the same word may carry different sentiment in different fields. Coverage gaps remain a concern, particularly for specialized terminology, neologisms, and code-switched text.

Compared to supervised machine learning approaches such as Naive Bayes or logistic regression, lexicon-based methods tend to achieve lower accuracy on well-defined benchmark tasks. However, they remain competitive in low-resource settings and provide a useful baseline against which data-driven models can be evaluated.

### Conclusion

Lexicon-based sentiment analysis offers an accessible, interpretable, and data-efficient approach to opinion mining. Its reliance on curated dictionaries and transparent aggregation rules makes it well-suited for exploratory analysis and domains where labeled data is unavailable. While supervised and deep learning methods have surpassed lexicon-based approaches on many benchmarks, the simplicity and explainability of the lexicon paradigm ensure its continued relevance in both research and applied NLP.

### References

Baccianella, S., Esuli, A., & Sebastiani, F. (2010). SentiWordNet 3.0: An enhanced lexical resource for sentiment analysis and opinion mining. *Proceedings of the Seventh International Conference on Language Resources and Evaluation (LREC'10)*, 2200-2204.

Hutto, C. J., & Gilbert, E. (2014). VADER: A parsimonious rule-based model for sentiment analysis of social media text. *Proceedings of the International AAAI Conference on Web and Social Media, 8*(1), 216-225.

Liu, B. (2012). Sentiment analysis and opinion mining. *Synthesis Lectures on Human Language Technologies, 5*(1), 1-167.

Loughran, T., & McDonald, B. (2011). When is a liability not a liability? Textual analysis, dictionaries, and 10-Ks. *The Journal of Finance, 66*(1), 35-65.

Nielsen, F. Å. (2011). A new ANEW: Evaluation of a word list for sentiment analysis in microblogs. *Proceedings of the ESWC2011 Workshop on 'Making Sense of Microposts'*, 93-98.

Salathé, M., & Khandelwal, S. (2011). Assessing vaccination sentiments with online social media: Implications for infectious disease dynamics and control. *PLoS Computational Biology, 7*(10), e1002199.

Taboada, M., Brooke, J., Tofiloski, M., Voll, K., & Stede, M. (2011). Lexicon-based methods for sentiment analysis. *Computational Linguistics, 37*(2), 267-307.

---

# Part 2: Cyberbullying Classification

## 1. Data Preparation

In [ ]:
# --- Import Libraries ---
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Scikit-learn: vectorization, model selection, classifiers, and evaluation metrics
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Download required NLTK resources (quiet=True suppresses download messages)
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

### 1.1 Load and Explore the Dataset

In [ ]:
# Load the dataset from CSV file
df = pd.read_csv("data/CyberBullying Comments Dataset.csv")

# Basic data exploration: shape, column names, data types, and missing values
print(f"Dataset shape: {df.shape}")
print(f"\nColumn names: {df.columns.tolist()}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")

# Display the first 5 rows to understand the data structure
print(f"\nFirst 5 rows:")
df.head()

In [ ]:
# Check class distribution to assess potential data imbalance
print("Class distribution:")
print(df['CB_Label'].value_counts())
print(f"\nClass proportions:")
print(df['CB_Label'].value_counts(normalize=True))

# Visualize class balance using a bar chart
fig, ax = plt.subplots(figsize=(6, 4))
df['CB_Label'].value_counts().plot(kind='bar', ax=ax, color=['steelblue', 'coral'])
ax.set_title('Class Distribution')
ax.set_xlabel('Label (0 = Not Cyberbullying, 1 = Cyberbullying)')
ax.set_ylabel('Count')
ax.set_xticklabels(['Not Cyberbullying (0)', 'Cyberbullying (1)'], rotation=0)
plt.tight_layout()
plt.show()

# Observation: the dataset is perfectly balanced (50/50 split)
print("\nThe dataset is perfectly balanced with 5,550 samples per class.")

### 1.2 Text Preprocessing

In [ ]:
# Define the set of English stopwords for filtering
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    """Clean and preprocess a text string for classification."""
    # Step 1: Convert to lowercase for consistency
    text = text.lower()
    # Step 2: Remove URLs (http/https and www links)
    text = re.sub(r'http\S+|www\S+', '', text)
    # Step 3: Remove mentions (@user) and hashtags (#topic)
    text = re.sub(r'@\w+|#\w+', '', text)
    # Step 4: Remove non-alphabetic characters (numbers, punctuation, etc.)
    text = re.sub(r'[^a-z\s]', '', text)
    # Step 5: Tokenize text into individual words using NLTK word_tokenize
    tokens = word_tokenize(text)
    # Step 6: Remove stopwords and tokens shorter than 3 characters
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    # Rejoin tokens into a single cleaned string
    return ' '.join(tokens)

# Apply preprocessing to the entire 'Text' column
df['clean_text'] = df['Text'].astype(str).apply(preprocess_text)

# Display sample results to verify the preprocessing pipeline
print("Sample preprocessing results:")
for i in range(3):
    print(f"\nOriginal:  {df['Text'].iloc[i][:100]}")
    print(f"Cleaned:   {df['clean_text'].iloc[i][:100]}")

### 1.3 TF-IDF Vectorization and Train/Test Split

In [ ]:
# --- TF-IDF Vectorization ---
# TF-IDF (Term Frequency-Inverse Document Frequency) converts text into numerical features.
# max_features=10000 limits the vocabulary to the top 10,000 most informative terms.
# TF-IDF is preferred over raw Bag-of-Words as it down-weights common terms
# and highlights distinctive words, which helps both NB and LR classifiers.
tfidf = TfidfVectorizer(max_features=10000)
X = tfidf.fit_transform(df['clean_text'])
y = df['CB_Label']

print(f"TF-IDF matrix shape: {X.shape}")
print(f"Number of features: {X.shape[1]}")

# --- Train/Test Split ---
# 80/20 split as required by the assignment instructions.
# stratify=y ensures both classes are equally represented in train and test sets.
# random_state=42 for reproducibility.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
print(f"\nTraining label distribution:\n{y_train.value_counts()}")
print(f"\nTest label distribution:\n{y_test.value_counts()}")

## 2. Naive Bayes Classification

Naive Bayes is a generative classifier based on Bayes' theorem with a strong (naive) independence assumption between features. It models P(d|c) and P(c) to compute P(c|d) via the MAP rule: C_MAP = argmax P(d|c) * P(c).

MultinomialNB is chosen because TF-IDF produces non-negative feature values representing word frequencies/importance, which satisfies the input requirement of the multinomial variant. This variant is best suited for document classification with word count or frequency-based features (Lecture 04).

In [ ]:
# --- Train Multinomial Naive Bayes ---
# MultinomialNB uses word frequencies/TF-IDF scores as features.
# It computes class-conditional probabilities P(word|class) for each word
# and combines them using the naive independence assumption.
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

# Generate predictions on the held-out test set
nb_preds = nb_model.predict(X_test)

# --- Evaluate using accuracy, precision, recall, and F1-score ---
# Precision: of all predicted cyberbullying, how many are correct? (TP / (TP + FP))
# Recall: of all actual cyberbullying, how many did we catch? (TP / (TP + FN))
# F1-score: harmonic mean of precision and recall, balancing both metrics
print("=== Naive Bayes Results ===\n")
print(f"Accuracy:  {accuracy_score(y_test, nb_preds):.4f}")
print(f"Precision: {precision_score(y_test, nb_preds):.4f}")
print(f"Recall:    {recall_score(y_test, nb_preds):.4f}")
print(f"F1-Score:  {f1_score(y_test, nb_preds):.4f}")
print(f"\nClassification Report:\n")
print(classification_report(y_test, nb_preds, target_names=['Not Cyberbullying', 'Cyberbullying']))

In [ ]:
# --- Confusion Matrix for Naive Bayes ---
# The confusion matrix shows TP, FP, FN, TN counts.
# Rows represent actual labels, columns represent predicted labels.
fig, ax = plt.subplots(figsize=(6, 5))
cm_nb = confusion_matrix(y_test, nb_preds)
sns.heatmap(cm_nb, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Not CB', 'CB'], yticklabels=['Not CB', 'CB'])
ax.set_title('Naive Bayes - Confusion Matrix')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

## 3. Logistic Regression Classification

Logistic Regression is a discriminative classifier that directly models P(c|d) by learning a decision boundary between classes. Unlike Naive Bayes (a generative model), it does not model P(d|c) but instead uses the sigmoid function to map a linear combination of features to a probability in (0, 1). The weights are optimized via Maximum Likelihood Estimation (MLE) using cross-entropy loss (Lecture 05).

In [ ]:
# --- Train Logistic Regression ---
# Logistic Regression applies the sigmoid function: P(y=1|x) = 1 / (1 + e^(-z))
# where z = b + w1*x1 + ... + wn*xn is the linear combination of TF-IDF features.
# max_iter=1000 ensures convergence for high-dimensional TF-IDF data.
# random_state=42 for reproducibility.
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

# Generate predictions on the same held-out test set used for Naive Bayes
lr_preds = lr_model.predict(X_test)

# --- Evaluate using the same metrics as Naive Bayes for fair comparison ---
print("=== Logistic Regression Results ===\n")
print(f"Accuracy:  {accuracy_score(y_test, lr_preds):.4f}")
print(f"Precision: {precision_score(y_test, lr_preds):.4f}")
print(f"Recall:    {recall_score(y_test, lr_preds):.4f}")
print(f"F1-Score:  {f1_score(y_test, lr_preds):.4f}")
print(f"\nClassification Report:\n")
print(classification_report(y_test, lr_preds, target_names=['Not Cyberbullying', 'Cyberbullying']))

In [ ]:
# --- Confusion Matrix for Logistic Regression ---
# Same layout as the NB confusion matrix for visual comparison.
fig, ax = plt.subplots(figsize=(6, 5))
cm_lr = confusion_matrix(y_test, lr_preds)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Oranges', ax=ax,
            xticklabels=['Not CB', 'CB'], yticklabels=['Not CB', 'CB'])
ax.set_title('Logistic Regression - Confusion Matrix')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

## 4. Model Comparison Summary

In [ ]:
# --- Side-by-side comparison table ---
# Aggregate all four metrics for both models into a single DataFrame
results = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Naive Bayes': [
        accuracy_score(y_test, nb_preds),
        precision_score(y_test, nb_preds),
        recall_score(y_test, nb_preds),
        f1_score(y_test, nb_preds)
    ],
    'Logistic Regression': [
        accuracy_score(y_test, lr_preds),
        precision_score(y_test, lr_preds),
        recall_score(y_test, lr_preds),
        f1_score(y_test, lr_preds)
    ]
})
results[['Naive Bayes', 'Logistic Regression']] = results[['Naive Bayes', 'Logistic Regression']].round(4)
print("=== Model Comparison ===\n")
print(results.to_string(index=False))

# --- Bar chart for visual comparison of all metrics ---
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(results['Metric']))
width = 0.35
ax.bar(x - width/2, results['Naive Bayes'], width, label='Naive Bayes', color='steelblue')
ax.bar(x + width/2, results['Logistic Regression'], width, label='Logistic Regression', color='coral')
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison')
ax.set_xticks(x)
ax.set_xticklabels(results['Metric'])
ax.legend()
ax.set_ylim(0.5, 1.0)
plt.tight_layout()
plt.show()

## 4. Short Report: Model Comparison

### Performance Overview

Both Naive Bayes (MultinomialNB) and Logistic Regression were trained on TF-IDF features extracted from the CyberBullying Comments Dataset (11,100 samples, perfectly balanced). The table and chart above present accuracy, precision, recall, and F1-score on the held-out 20% test set.

Both models achieve comparable performance, with Naive Bayes holding a slight edge across accuracy (0.72 vs. 0.71), recall (0.68 vs. 0.65), and F1-score (0.71 vs. 0.69). Precision is nearly identical for both models (~0.74). This result suggests that the conditional independence assumption of Naive Bayes is not a major liability on this dataset, where individual word presence (e.g., slurs, insults) is highly indicative of cyberbullying, and word co-occurrence patterns add limited discriminative power beyond unigram features.

A key distinction between the two models lies in their classification approach. Naive Bayes is a **generative classifier** that models P(d|c) and P(c) separately, then applies Bayes' rule to compute P(c|d). Logistic Regression is a **discriminative classifier** that directly learns P(c|d) by optimizing a decision boundary via the sigmoid function and MLE. In theory, discriminative classifiers can outperform generative ones given sufficient data, but on this balanced, unigram-driven dataset, both approaches perform similarly.

### Advantages and Disadvantages

**Naive Bayes**

| | |
|---|---|
| **Advantage 1** | Very fast to train and predict, making it practical for large-scale or real-time content moderation. |
| **Advantage 2** | Performs well even with limited training data due to its strong prior assumptions, which act as implicit regularization. On this dataset, its simplicity proved sufficient to match or exceed the discriminative model. |
| **Disadvantage 1** | The conditional independence assumption ignores word co-occurrence and phrase-level patterns, which limits its ability to capture nuanced cyberbullying expressions (e.g., sarcasm or coded language). |
| **Disadvantage 2** | Produces poorly calibrated probability estimates, which makes threshold tuning for content moderation applications less reliable. |

**Logistic Regression**

| | |
|---|---|
| **Advantage 1** | Directly optimizes the decision boundary, which generally leads to better performance on complex text classification tasks where feature interactions matter. |
| **Advantage 2** | Produces well-calibrated probability estimates, useful for adjusting classification thresholds in sensitive applications like content moderation. |
| **Disadvantage 1** | Slightly lower recall on this dataset (0.65 vs. 0.68), meaning it misses more actual cyberbullying comments. In a content moderation setting, false negatives are costly. |
| **Disadvantage 2** | Slower to train than Naive Bayes and requires tuning of regularization strength. On a balanced, unigram-driven dataset like this one, the additional complexity does not translate to better performance. |

### Additional Findings

The dataset is perfectly balanced (50/50 split), eliminating class imbalance as a confounding factor and making accuracy a reliable metric alongside F1-score. Both models achieve around 72% accuracy, which indicates that cyberbullying detection from text alone is a moderately difficult task. Many comments likely contain ambiguous language where context, tone, or sarcasm determines whether the text is bullying.

The similar performance of both models suggests that TF-IDF unigram features capture the bulk of the signal in this dataset. Cyberbullying comments tend to contain distinctive vocabulary (slurs, profanity, aggressive terms) that both models detect effectively through individual word weights. Improving beyond this baseline would likely require richer feature representations such as n-grams, word embeddings, or transformer-based models that capture sequential and contextual meaning.